In [1]:
from ultralytics import YOLO
import cv2
import csv
import time

In [34]:
from ultralytics import YOLO
import cv2
import csv

model = YOLO("yolo26s-pose.pt")

cap = cv2.VideoCapture(1)

BODY_PARTS = [
    "Nose",
    "Left Eye",
    "Right Eye",
    "Left Ear",
    "Right Ear",
    "Left Shoulder",
    "Right Shoulder",
    "Left Elbow",
    "Right Elbow",
    "Left Wrist",
    "Right Wrist",
    "Left Hip",
    "Right Hip",
    "Left Knee",
    "Right Knee",
    "Left Ankle",
    "Right Ankle"
]

# joints you want to REMOVE
exclude = {0, 1, 2, 3, 4}

# joints you want to KEEP
keep_indices = [i for i in range(len(BODY_PARTS)) if i not in exclude]

file = open("data/detections.csv", "a", newline="")
writer = csv.writer(file)

# CSV header
header = ["frame"]
for i in keep_indices:
    part = BODY_PARTS[i]
    header += [f"{part}_x", f"{part}_y", f"{part}_conf"]

writer.writerow(header)

frame_id = 0
height = 1080
width = 1920

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    annotated = results[0].plot()

    keypoints = results[0].keypoints

    if keypoints is not None:
        xy = keypoints.x
        conf = keypoints.conf

        for person_idx in range(len(xy)):
            row = [frame_id]

            for i in keep_indices:
                x = xy[person_idx][i][0].item() / width
                y = xy[person_idx][i][1].item() / height
                c = conf[person_idx][i].item()

                row += [x, y, c]

            writer.writerow(row)
            file.flush()

    frame_id += 1

    cv2.imshow("YOLO Pose", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
file.close()
cv2.destroyAllWindows()

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-pose.pt... <urlopen error [Errno 60] Operation timed out>


KeyboardInterrupt: 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn

In [30]:
df = pd.read_csv('data/detections.csv')

df.head()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422


In [27]:
df.describe()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
count,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,...,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000
mean,31.246575,1479.845210,996.943710,0.769920,812.329511,1019.888796,0.806980,1607.940868,1047.239123,0.010139,...,0.001645,950.307367,1044.377972,0.001834,1299.571025,1057.897777,0.000889,1050.509838,1059.338103,0.001494
std,16.641465,109.220648,45.611458,0.263498,314.004251,38.465325,0.268999,113.336366,43.296435,0.019162,...,0.000809,239.105332,50.849688,0.001673,176.121766,34.347196,0.001602,230.457615,33.501083,0.003552
min,0.000000,1386.805908,788.172791,0.002668,690.666321,807.235657,0.004874,1470.353638,851.452148,0.001042,...,0.000724,815.877380,890.705933,0.000454,1167.926147,957.878723,0.000151,921.878235,953.080811,0.000327
25%,18.000000,1429.003174,980.209961,0.800902,702.842834,1024.550903,0.831938,1511.337036,1028.748779,0.001792,...,0.001135,835.933655,1022.260315,0.001100,1181.574951,1042.947144,0.000240,933.222900,1053.443481,0.000438
50%,32.000000,1439.810303,1007.847717,0.859499,712.725769,1030.075195,0.907504,1530.315918,1065.807129,0.003653,...,0.001362,863.668945,1080.000000,0.001483,1206.568359,1080.000000,0.000388,950.536621,1080.000000,0.000673
75%,45.000000,1473.755615,1026.478271,0.905596,721.202454,1033.263916,0.938942,1697.711060,1080.000000,0.010990,...,0.001947,926.695740,1080.000000,0.001672,1378.994873,1080.000000,0.001278,1069.377319,1080.000000,0.001288
max,59.000000,1866.227539,1050.137329,0.988900,1858.493042,1040.435791,0.962918,1876.223267,1080.000000,0.146377,...,0.004756,1857.587891,1080.000000,0.010352,1912.609131,1080.000000,0.012873,1899.409058,1080.000000,0.029463


In [28]:
df.shape

(73, 37)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   frame                73 non-null     int64  
 1   Left Shoulder_x      73 non-null     float64
 2   Left Shoulder_y      73 non-null     float64
 3   Left Shoulder_conf   73 non-null     float64
 4   Right Shoulder_x     73 non-null     float64
 5   Right Shoulder_y     73 non-null     float64
 6   Right Shoulder_conf  73 non-null     float64
 7   Left Elbow_x         73 non-null     float64
 8   Left Elbow_y         73 non-null     float64
 9   Left Elbow_conf      73 non-null     float64
 10  Right Elbow_x        73 non-null     float64
 11  Right Elbow_y        73 non-null     float64
 12  Right Elbow_conf     73 non-null     float64
 13  Left Wrist_x         73 non-null     float64
 14  Left Wrist_y         73 non-null     float64
 15  Left Wrist_conf      73 non-null     float

In [31]:
df.max()

frame                    59.000000
Left Shoulder_x        1866.227539
Left Shoulder_y        1050.137329
Left Shoulder_conf        0.988900
Right Shoulder_x       1858.493042
Right Shoulder_y       1040.435791
Right Shoulder_conf       0.962918
Left Elbow_x           1876.223267
Left Elbow_y           1080.000000
Left Elbow_conf           0.146377
Right Elbow_x          1868.676392
Right Elbow_y          1080.000000
Right Elbow_conf          0.018749
Left Wrist_x           1920.000000
Left Wrist_y           1064.460571
Left Wrist_conf           0.783444
Right Wrist_x          1904.329468
Right Wrist_y          1065.752563
Right Wrist_conf          0.415467
Left Hip_x             1879.369385
Left Hip_y             1080.000000
Left Hip_conf             0.003477
Right Hip_x            1883.385986
Right Hip_y            1080.000000
Right Hip_conf            0.005730
Left Knee_x            1856.340088
Left Knee_y            1080.000000
Left Knee_conf            0.004756
Right Knee_x        

In [32]:
df.head()

,frame,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422


In [33]:
df = df.drop(columns='frame')
df.head()

,Left Shoulder_x,Left Shoulder_y,Left Shoulder_conf,Right Shoulder_x,Right Shoulder_y,Right Shoulder_conf,Left Elbow_x,Left Elbow_y,Left Elbow_conf,Right Elbow_x,...,Left Knee_conf,Right Knee_x,Right Knee_y,Right Knee_conf,Left Ankle_x,Left Ankle_y,Left Ankle_conf,Right Ankle_x,Right Ankle_y,Right Ankle_conf
0,1424.110962,993.762634,0.905596,704.344666,1020.448792,0.913182,1518.191406,1071.431396,0.005509,613.872437,...,0.001585,838.337830,1080.0,0.001660,1192.874146,1080.0,0.000352,940.914551,1080.0,0.000696
1,1438.859985,1014.715759,0.933951,697.710754,1028.207520,0.940646,1517.736938,1080.000000,0.002958,634.289734,...,0.001362,828.336609,1080.0,0.001547,1180.128418,1080.0,0.000240,928.343689,1080.0,0.000516
2,1446.010254,1031.935547,0.923083,698.586304,1031.863037,0.936629,1518.396851,1080.000000,0.002013,639.363098,...,0.001350,817.569214,1080.0,0.001726,1171.809814,1080.0,0.000203,922.086548,1080.0,0.000456
3,1439.775635,1035.010986,0.920916,702.842834,1031.151489,0.937211,1517.457397,1080.000000,0.001792,645.795166,...,0.001236,817.365417,1080.0,0.001657,1167.926147,1080.0,0.000184,924.311829,1080.0,0.000417
4,1439.810303,1035.710205,0.920352,700.170227,1030.109497,0.935427,1519.770630,1080.000000,0.001756,643.418335,...,0.001230,815.877380,1080.0,0.001672,1170.679321,1080.0,0.000182,924.766174,1080.0,0.000422
